In [1]:
from pathlib import Path
from retinaface import RetinaFace
import cv2
import os

I0000 00:00:1790613674.707529 1064802 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790613674.710246 1064802 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790613674.793763 1064802 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790613675.995463 1064802 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

In [3]:
input_folder = "raw_data/images"
output_folder = "raw_data/extracted_faces"

In [5]:
for image_file in Path(input_folder).glob("*"):
    if image_file.suffix.lower() not in [".jpg", ".jpeg", ".png", ".webp"]:
        continue

    try:
        faces = RetinaFace.extract_faces(
            img_path=str(image_file),
            align=True
        )

        for idx, face in enumerate(faces):
            face = cv2.cvtColor(face, cv2.COLOR_RGB2BGR)

            filename = f"{image_file.stem}_face_{idx+1}.jpg"
            cv2.imwrite(
                os.path.join(output_folder, filename),
                face
            )

        print(f"{image_file.name}: {len(faces)} faces")

    except Exception as e:
        print(f"Error: {image_file.name} -> {e}")

E0000 00:00:1790613820.168377 1064802 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


IMG_5331.jpg: 1 faces
IMG_5347.jpg: 1 faces
IMG_5320.jpg: 1 faces
IMG_5327.jpg: 1 faces
IMG_5302.jpg: 1 faces
IMG_5341.jpg: 1 faces
IMG_5306.jpg: 1 faces
IMG_5307.jpg: 1 faces
IMG_5350.jpg: 1 faces
IMG_5334.jpg: 1 faces
IMG_5308.jpg: 1 faces
IMG_5305.jpg: 1 faces
IMG_5336.jpg: 1 faces
IMG_5329.jpg: 1 faces
IMG_5314.jpg: 1 faces
IMG_5317.jpg: 1 faces
IMG_5337.jpg: 1 faces
IMG_5339.jpg: 1 faces
IMG_5340.jpg: 1 faces
IMG_5342.jpg: 1 faces
IMG_5333.jpg: 1 faces
IMG_5321.jpg: 1 faces
IMG_5349.jpg: 1 faces
IMG_5312.jpg: 1 faces
IMG_5326.jpg: 1 faces
IMG_5313.jpg: 1 faces
IMG_5315.jpg: 1 faces
IMG_5348.jpg: 1 faces
IMG_5304.jpg: 1 faces
IMG_5318.jpg: 1 faces
IMG_5319.jpg: 1 faces
IMG_5325.jpg: 1 faces
IMG_5311.jpg: 1 faces
IMG_5345.jpg: 1 faces
IMG_5335.jpg: 1 faces
IMG_5351.jpg: 1 faces
IMG_5316.jpg: 1 faces
IMG_5323.jpg: 1 faces
IMG_5310.jpg: 1 faces
IMG_5303.jpg: 1 faces
IMG_5332.jpg: 2 faces
IMG_5344.jpg: 1 faces
IMG_5352.jpg: 1 faces
IMG_5338.jpg: 1 faces
IMG_5343.jpg: 1 faces
IMG_5328.j

#### **Generate ArcFace Embeddings from Face Images**

The InsightFace/ArcFace pipeline exposes normalized embeddings that can be directly compared with cosine similarity.

In [6]:
import cv2
import numpy as np
from pathlib import Path
from insightface.app import FaceAnalysis

# ArcFace Model (buffalo_l)
app = FaceAnalysis(
    name="buffalo_l",
    providers=["CPUExecutionProvider"]
)

app.prepare(ctx_id=0)

faces_dir = "raw_data/images"

for image_file in Path(faces_dir).glob("*.jpg"):
    image = cv2.imread(str(image_file))

    detected_faces = app.get(image)

    if not detected_faces:
        print(f"No face found: {image_file}")
        continue

    face = detected_faces[0]

    # 512-dimensional embedding
    embedding = face.embedding

    print(f"\n{image_file.name}")
    print(f"Embedding Shape: {embedding.shape}")
    print(f"First 10 Values: {embedding[:10]}")


IMG_5331.jpg
Embedding Shape: (512,)
First 10 Values: [-0.82762647 -1.214981   -0.42978555 -0.17795485 -0.7973765   1.1482568
  1.798462   -0.673821   -0.31838465  0.9779494 ]

IMG_5347.jpg
Embedding Shape: (512,)
First 10 Values: [-0.00631276  0.20891912  0.67266625  2.1345384  -1.258031    1.7311844
  1.5323912  -0.47794276  1.3731738  -0.83320147]

IMG_5320.jpg
Embedding Shape: (512,)
First 10 Values: [-0.28793368 -0.6982314   1.8496642  -1.9421737  -0.39822125  0.27153626
  0.6846997   0.7576397   1.5923162   0.41566402]

IMG_5327.jpg
Embedding Shape: (512,)
First 10 Values: [ 0.51405346 -1.5414001   0.91890985  0.5551709  -0.11119585  1.0704643
  0.6080945  -0.3635045   1.2710088  -0.20107234]

IMG_5302.jpg
Embedding Shape: (512,)
First 10 Values: [ 0.00679733 -0.21034859 -0.4783312   1.7466309   1.6011213  -1.0102533
  0.86561406  1.6820023  -0.19425087  0.31361312]

IMG_5341.jpg
Embedding Shape: (512,)
First 10 Values: [ 1.045077   -1.4230897  -0.36435008 -0.02151688  0.2425592

#### **Save Embeddings to Numpy Files**

In [7]:
import cv2
import numpy as np
from pathlib import Path
from insightface.app import FaceAnalysis

app = FaceAnalysis(
    name="buffalo_l",
    providers=["CPUExecutionProvider"]
)

app.prepare(ctx_id=0)

# faces_dir = "raw_data/images"
faces_dir = Path("raw_data/images")
embeddings_dir = Path("raw_data/embeddings")

embeddings_dir.mkdir(exist_ok=True)

for image_file in faces_dir.glob("*.jpg"):

    image = cv2.imread(str(image_file))

    faces = app.get(image)

    if not faces:
        continue

    embedding = faces[0].embedding

    np.save(
        embeddings_dir / f"{image_file.stem}.npy",
        embedding
    )

    print(f"Saved {image_file.stem}.npy")

Saved IMG_5331.npy
Saved IMG_5347.npy
Saved IMG_5320.npy
Saved IMG_5327.npy
Saved IMG_5302.npy
Saved IMG_5341.npy
Saved IMG_5306.npy
Saved IMG_5307.npy
Saved IMG_5350.npy
Saved IMG_5334.npy
Saved IMG_5308.npy
Saved IMG_5305.npy
Saved IMG_5336.npy
Saved IMG_5329.npy
Saved IMG_5314.npy
Saved IMG_5317.npy
Saved IMG_5337.npy
Saved IMG_5339.npy
Saved IMG_5340.npy
Saved IMG_5342.npy
Saved IMG_5333.npy
Saved IMG_5321.npy
Saved IMG_5349.npy
Saved IMG_5312.npy
Saved IMG_5326.npy
Saved IMG_5313.npy
Saved IMG_5315.npy
Saved IMG_5348.npy
Saved IMG_5304.npy
Saved IMG_5318.npy
Saved IMG_5319.npy
Saved IMG_5325.npy
Saved IMG_5311.npy
Saved IMG_5345.npy
Saved IMG_5335.npy
Saved IMG_5351.npy
Saved IMG_5316.npy
Saved IMG_5323.npy
Saved IMG_5310.npy
Saved IMG_5303.npy
Saved IMG_5332.npy
Saved IMG_5344.npy
Saved IMG_5352.npy
Saved IMG_5338.npy
Saved IMG_5343.npy
Saved IMG_5328.npy
Saved IMG_5330.npy
Saved IMG_5324.npy
Saved IMG_5346.npy


#### **Compare Two Faces**

ArcFace embeddings are designed for cosine-similarity matching in recognition, search, and watchlist workflows.

In [18]:
import numpy as np

emb1 = np.load("raw_data/embeddings/IMG_5302.npy")
emb2 = np.load("raw_data/embeddings/IMG_5315.npy")

# Cosine Similarity
similarity = np.dot(
    emb1 / np.linalg.norm(emb1),
    emb2 / np.linalg.norm(emb2)
)

print(f"Similarity: {similarity:.4f}")

if similarity > 0.5:
    print("Same Person")
else:
    print("Different Person")

Similarity: 0.0045
Different Person


#### **How to use FAISS**

Step 1: Generate ArcFace Embeddings

In [19]:
emb1.shape

(512,)

In [65]:
# Step 2: Create a FAISS Index
import faiss
import numpy as np

DIMENSION = 512

# Cosine similarity requires normalized vectors
index = faiss.IndexFlatIP(DIMENSION)

print(index.ntotal)

0


In [47]:
# Step 3: Add Face Embeddings
import numpy as np

embeddings = np.array([
    emb1,
    emb2,
    # emb3
], dtype=np.float32)

# Normalize
faiss.normalize_L2(embeddings)

index.add(embeddings)

print(index.ntotal)

16


In [66]:
def create_embedding(file_path=False):
    embedding = np.array([
            np.load(file_path),
        ], dtype=np.float32)
    
    # Normalize
    faiss.normalize_L2(embedding)

    return embedding
    

# def add_embeddings(emb=False):    
#     embeddings = np.array([
#         emb,
#     ], dtype=np.float32)

#     # Normalize
#     faiss.normalize_L2(embeddings)

#     index.add(embeddings)

def add_embeddings(file_path=False):    
    embedding = create_embedding(file_path=file_path)

    index.add(embedding)

In [67]:
from pathlib import Path

dir_path = Path("raw_data/embeddings")

emb_files = [f for f in dir_path.rglob("*.npy") if f.is_file()]

for file in emb_files:
    print(file)  # Prints the full path relative to dir_path
    # add_embeddings(emb=os.fspath(file))
    # emb = np.load(file)
    # add_embeddings(emb=emb)
    add_embeddings(file_path=file)
    
print(index.ntotal)

raw_data/embeddings/IMG_5302.npy
raw_data/embeddings/IMG_5337.npy
raw_data/embeddings/IMG_5303.npy
raw_data/embeddings/IMG_5320.npy
raw_data/embeddings/IMG_5326.npy
raw_data/embeddings/IMG_5325.npy
raw_data/embeddings/IMG_5330.npy
raw_data/embeddings/IMG_5310.npy
raw_data/embeddings/IMG_5311.npy
raw_data/embeddings/IMG_5308.npy
raw_data/embeddings/IMG_5328.npy
raw_data/embeddings/IMG_5336.npy
raw_data/embeddings/IMG_5312.npy
raw_data/embeddings/IMG_5319.npy
raw_data/embeddings/IMG_5347.npy
raw_data/embeddings/IMG_5349.npy
raw_data/embeddings/IMG_5348.npy
raw_data/embeddings/IMG_5321.npy
raw_data/embeddings/IMG_5307.npy
raw_data/embeddings/IMG_5317.npy
raw_data/embeddings/IMG_5318.npy
raw_data/embeddings/IMG_5335.npy
raw_data/embeddings/IMG_5316.npy
raw_data/embeddings/IMG_5352.npy
raw_data/embeddings/IMG_5340.npy
raw_data/embeddings/IMG_5342.npy
raw_data/embeddings/IMG_5323.npy
raw_data/embeddings/IMG_5350.npy
raw_data/embeddings/IMG_5331.npy
raw_data/embeddings/IMG_5304.npy
raw_data/e

#### **Step 4: Store Person Mapping**

FAISS stores vectors only.

Keep a mapping:

In [62]:
person_mapping = {
    0: {
        "person_id": 101,
        "name": "John"
    },
    1: {
        "person_id": 102,
        "name": "Alice"
    },
}

In [68]:
# Step 5: Search a Face
# emb = np.load('raw_data/embeddings/IMG_5338.npy')

query = np.array(
    create_embedding(file_path='raw_data/embeddings/IMG_5338.npy'),
    dtype=np.float32
)

faiss.normalize_L2(query)

k = 5

distances, indices = index.search(
    query,
    k
)

print(distances)
print(indices)

[[1.         0.8851398  0.80934083 0.76927066 0.74884254]]
[[48  1 40 24 11]]


#### **Step 6: Match Decision**

| Similarity | Meaning                  |
| ---------- | ------------------------ |
| > 0.80     | Very likely same person  |
| 0.65–0.80  | Possible match           |
| < 0.60     | Usually different person |


In [69]:
best_score = distances[0][0]
best_idx = indices[0][0]

if best_score > 0.6:
    print("Known Person")
else:
    print("Unknown Person")

Known Person


#### **Step 7: Save/Load FAISS Index**

In [70]:
faiss.write_index(
    index,
    "Datasets/faces.index"
)

In [71]:
index = faiss.read_index(
    "Datasets/faces.index"
)

#### **Complete Registration Function**

In [ ]:
def register_face(embedding, person_id):
    embedding = np.array(
        [embedding],
        dtype=np.float32
    )

    faiss.normalize_L2(embedding)

    index.add(embedding)

    return index.ntotal - 1

#### **Complete Search Function**

In [ ]:
def search_face(embedding, threshold=0.6):
    query = np.array(
        [embedding],
        dtype=np.float32
    )

    faiss.normalize_L2(query)

    distances, indices = index.search(
        query,
        1
    )

    score = float(distances[0][0])
    idx = int(indices[0][0])

    if score >= threshold:
        return idx, score

    return None, score